# Module 1 — Rigorous Correlation Analysis

**By the end of this notebook, you will be able to:**
- Read a correlation coefficient together with its p-value, and know why neither is enough alone
- Avoid a common correlation-analysis bug: computing p-values pairwise instead of dropping rows across every column at once
- Build and interpret a correlation heatmap that only shows statistically significant relationships

**Context:** In the discovery notebook (Session 2), you saw a first, quick correlation heatmap on Kampala and Nairobi only. Here, you do it properly: on the full dataset (Kampala, Nairobi, Lagos, and Bujumbura), with Pearson **and** Spearman correlation, their p-values, and a heatmap that only shows what is actually reliable.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]
COLUMNS = [
    "city", "date", "hour", "site_latitude", "site_longitude", "pm2_5",
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]

df = pd.read_csv("../../data/train.csv")
df = df[df["city"].isin(CITIES)][COLUMNS].copy()
df["date"] = pd.to_datetime(df["date"])
df["city"].value_counts()

city
Kampala      5596
Nairobi      1500
Lagos         852
Bujumbura     123
Name: count, dtype: int64

## Correlation with `pm2_5`

A single correlation number is not enough:

- **Pearson's r** measures *linear* correlation and is sensitive to outliers.
- **Spearman's rho** measures *monotonic* correlation using ranks, so it is more robust to outliers and non-linear (but still monotonic) relationships.
- The **p-value** estimates how likely it would be to observe a correlation this strong by chance alone if the true correlation were zero. A coefficient without a p-value tells you nothing about whether it could just be noise — and a low p-value does not mean the relationship is strong, only that it is unlikely to be pure chance. Neither implies causation.

For a deeper treatment of p-values — common misconceptions, how to read them alongside the coefficient, when they can mislead — see [section 7 of Machine Learning Techniques: From Theory to Practice](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#7-focus-on-p-value).

**Documentation references:**
- [`scipy.stats.pearsonr()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html)
- [`scipy.stats.spearmanr()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)
- [`DataFrame.dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)

**Exercise:** Write a function that computes, for one column against `pm2_5`, both Pearson and Spearman correlation together with their p-values. Before correlating, restrict the data to rows where **both of those two columns** (not all 8) have a value — `DataFrame.dropna()` accepts a `subset` of columns to check, so you don't have to select the columns manually first. This matters because `uvaerosollayerheight_aerosol_height` alone is missing over 90% of the time: requiring all 8 columns to be present simultaneously would collapse the usable sample and bias every correlation, not just that one variable's.

In [2]:
# TODO 1: define a function that computes Pearson and Spearman correlation (with p-values)
# between one column and the target, dropping missing values for that pair only
def correlate_with_target(df, column, target_col):
    """
    Compute Pearson and Spearman correlation between one column and the target,
    together with their p-values.

    Parameters:
    - df: DataFrame containing column and target_col
    - column: name of the column to correlate with the target
    - target_col: name of the target column (e.g. "pm2_5")

    Returns:
    - dict with n, pearson_r, pearson_p, spearman_r, spearman_p
    """


**Exercise:** Apply `correlate_with_target` to every satellite column against `pm2_5`, collect the results in a DataFrame, and sort it by the strength of the Pearson correlation (largest absolute value first).

In [3]:
# TODO 2: apply correlate_with_target to every satellite column, and sort the results


### From one correlation to a full matrix

`correlation_table` compares every variable to `pm2_5`, one at a time. A **correlation matrix** compares every variable to every other variable at once — useful, but it is symmetric: the correlation between A and B is identical to the one between B and A, so a full heatmap would show every relationship twice (once above the diagonal, once below), plus a diagonal of 1s that isn't informative (a variable is always perfectly correlated with itself).

To keep only the useful half, and only the statistically significant cells, we build a **mask**: a same-shaped table of `True`/`False` where `True` means "hide this cell." `seaborn.heatmap` accepts a `mask=` argument for exactly this. We'll build it in a few small steps, then use it.

**Exercise:** Write a function that returns both the correlation matrix and the matching p-value matrix for a set of columns. `DataFrame.corr()` gives you the correlation matrix in one call. For the p-values, reuse the idea from `correlate_with_target` — pairwise `dropna`, then `pearsonr` — but applied to every *pair* of columns this time (a loop inside a loop), not just each column against `pm2_5`.

In [4]:
# TODO 3: define a function that returns the correlation matrix and the matching
# p-value matrix for a set of columns, reusing the pairwise-dropna idea from
# correlate_with_target (TODO 1) but for every pair of columns, not just one
# column against the target
def pairwise_correlation_matrices(df, columns):
    """
    Compute the correlation matrix and the matching p-value matrix for a set of columns.

    Parameters:
    - df: DataFrame containing the columns
    - columns: list of column names to correlate with each other

    Returns:
    - corr: DataFrame, the Pearson correlation matrix
    - pvalue_matrix: DataFrame of the same shape, the matching p-values
      (1.0 on the diagonal, since a variable is not tested against itself)
    """


**Exercise:** Apply `pairwise_correlation_matrices` to `satellite_columns + ["pm2_5"]` to get `corr` and `pvalue_matrix`. Then build the mask in two parts and combine them with `|` (a cell is hidden if *either* is true): (1) `pvalue_matrix > 0.05` for the non-significant cells, and (2) the upper triangle, using [`numpy.triu`](https://numpy.org/doc/stable/reference/generated/numpy.triu.html) on a same-shaped array of `True` — check its `k` argument to exclude the diagonal itself (you want to hide the redundant mirror, not the variables' self-correlation row).

In [5]:
# TODO 4: apply pairwise_correlation_matrices, then build a mask that hides
# non-significant cells (p >= 0.05) and the redundant upper triangle


**Exercise:** Draw the masked correlation heatmap with `seaborn.heatmap`, passing your `mask`.

In [6]:
# TODO 5: draw the correlation heatmap, hiding masked cells


**Exercise:** Zoom in on the variable most correlated with `pm2_5` — the first row of `correlation_table`, already sorted — with a scatter plot and a regression line: [`seaborn.regplot`](https://seaborn.pydata.org/generated/seaborn.regplot.html). Include the Pearson r, its p-value, and the sample size `n` in the title, so the plot carries its own evidence.

In [7]:
# TODO 6: scatter plot + regression line for the most correlated variable, with its
# statistics in the title


## Wrap-up

Compare `correlation_table` to what you saw in the discovery notebook, which only used Kampala and Nairobi. Did any variable's apparent importance change now that Lagos and Bujumbura are included? Does the most correlated variable have a large enough `n` for you to trust it? Write down what changed and what you would investigate next.

_Your observations here._